# AlexNet on MNIST

**Question:** Implement the AlexNet architecture using a deep learning framework and apply it to the given
image classification task.

---

## What is AlexNet?
AlexNet was created by Alex Krizhevsky, Ilya Sutskever, and Geoffrey Hinton in **2012**.
It won the ImageNet competition (1.2 million images, 1000 classes) by a huge margin and kickstarted the modern deep learning era.

**Key innovations at the time:**
- First large CNN to use **ReLU** activations (much faster than Sigmoid/Tanh)
- Used **Dropout** to fight overfitting
- Trained on **GPUs** for the first time

## Architecture Overview
AlexNet has two parts: a feature extractor (5 conv layers) and a classifier (3 FC layers):
```
Input: 3 x 224 x 224

Feature Extractor
  Conv1: 64 filters, 11x11, stride 4  ->  64x55x55  -> ReLU -> MaxPool  ->  64x27x27
  Conv2: 192 filters,  5x5, pad 2     -> 192x27x27  -> ReLU -> MaxPool  -> 192x13x13
  Conv3: 384 filters,  3x3, pad 1     -> 384x13x13  -> ReLU
  Conv4: 256 filters,  3x3, pad 1     -> 256x13x13  -> ReLU
  Conv5: 256 filters,  3x3, pad 1     -> 256x13x13  -> ReLU -> MaxPool  -> 256x6x6

Classifier
  Flatten  ->  9216
  Dropout -> FC1: 9216 -> 4096
  Dropout -> FC2: 4096 -> 4096 -> ReLU
  FC3:            4096 ->   10  (one score per digit)
```

> **Why use AlexNet on MNIST?**
> MNIST (28x28 grayscale) is too small for AlexNet which was designed for 224x224 RGB images.
> We adapt it by resizing to 224x224 and duplicating the grayscale channel into 3 channels.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

## Step 1 - Load and Preprocess MNIST

MNIST is 28x28 grayscale, but AlexNet needs 224x224 RGB. We apply four transforms:

1. **`Resize((224,224))`** - scale each image up from 28x28 to 224x224
2. **`Grayscale(num_output_channels=3)`** - MNIST has 1 channel (black and white); AlexNet expects 3 channels (RGB). This duplicates the single channel 3 times so the shape matches. All 3 channels are identical.
3. **`ToTensor()`** - convert PIL image to float tensor, pixel values in [0, 1]
4. **`Normalize`** - shift pixel values to range [-1, 1] for more stable training

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),               # AlexNet was designed for 224x224 input
    transforms.Grayscale(num_output_channels=3), # Duplicate grayscale into 3 identical RGB channels
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize all 3 channels to [-1, 1]
])

train_loader = DataLoader(
    torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform),
    batch_size=64, shuffle=True
)
test_loader = DataLoader(
    torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform),
    batch_size=1000
)

## Step 2 - Define AlexNet

### `self.feats` - Feature Extractor
Five convolutional layers that progressively learn more complex visual patterns:
- **Conv1**: Large 11x11 filter with stride 4 - captures broad strokes, reduces spatial size quickly
- **Conv2**: 5x5 filter - captures medium-scale features like curves and corners
- **Conv3, 4, 5**: 3x3 filters stacked - capture fine details
- **MaxPool(3,2)**: Reduces spatial size, keeping only the dominant features

### `self.classifier` - Classifier
Three fully connected layers that map extracted features to digit class scores:
- **Dropout**: Randomly zeros neurons during training to prevent overfitting
- **FC1/FC2**: Large 4096-neuron layers to learn complex combinations of features
- **FC3**: Final layer - 10 output scores, one per digit (0 through 9)

In [ ]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Feature Extractor: 5 convolutional layers
        self.feats = nn.Sequential(
            # Conv1: 3 channels -> 64 feature maps, 11x11 filter, stride=4, pad=2
            # 224x224 -> 55x55 after conv -> 27x27 after MaxPool
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2), nn.ReLU(), nn.MaxPool2d(3, 2),

            # Conv2: 64 -> 192 feature maps, 5x5 filter
            # 27x27 -> 27x27 after conv (padding preserves size) -> 13x13 after MaxPool
            nn.Conv2d(64, 192, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(3, 2),

            # Conv3: 192 -> 384 feature maps - no pooling, stays 13x13
            nn.Conv2d(192, 384, kernel_size=3, padding=1), nn.ReLU(),

            # Conv4: 384 -> 256 feature maps - no pooling, stays 13x13
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),

            # Conv5: 256 -> 256 feature maps -> 13x13 after conv -> 6x6 after MaxPool
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(3, 2)
        )

        # Classifier: 3 fully connected layers
        # Input: 256 channels x 6x6 spatial = 9216 values
        self.classifier = nn.Sequential(
            nn.Dropout(),                      # Drop 50% of neurons randomly (default p=0.5)
            nn.Linear(256 * 6 * 6, 4096),      # 9216 -> 4096
            nn.Dropout(),
            nn.Linear(4096, 4096), nn.ReLU(),  # 4096 -> 4096
            nn.Linear(4096, 10)                # 4096 -> 10 class scores
        )

    def forward(self, x):
        x = self.feats(x)           # Extract features: 3x224x224 -> 256x6x6
        x = x.view(x.size(0), -1)  # Flatten: 256x6x6 -> 9216 (keep batch size, flatten the rest)
        x = self.classifier(x)      # Classify: 9216 -> 10
        return x

In [ ]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Use GPU if available
model     = AlexNet().to(device)
criterion = nn.CrossEntropyLoss()                      # Standard loss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)   # Adam adapts learning rate automatically

## Step 3 - Train

We only train for **1 epoch** because AlexNet is very large - resizing MNIST to 224x224 makes each batch slow on CPU.

We print a **running loss every 100 batches** so we can watch progress in real time.
The running loss resets to 0 every 100 batches - it shows how the model did on the most recent 100 batches only.

`enumerate(train_loader)` gives us both the batch index `i` and the data `(images, labels)` - useful for printing every 100 steps.

In [ ]:
for epoch in range(1):  # Only 1 epoch - AlexNet on 224x224 images is slow on CPU
    model.train()
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):  # i = batch index
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()                      # Clear previous gradients
        loss = criterion(model(images), labels)    # Forward pass + compute loss
        loss.backward()                            # Backpropagate
        optimizer.step()                           # Update weights

        running_loss += loss.item()

        if i % 100 == 0:  # Print every 100 batches
            print(f'Epoch {epoch+1}, Batch {i+1} | Running Loss: {running_loss:.3f}')
            running_loss = 0  # Reset so next print shows only next 100 batches

## Step 4 - Evaluate on Test Set

We run the model on 10,000 test images it has never seen during training.

- `model.eval()` turns off Dropout so predictions are deterministic
- `torch.no_grad()` skips gradient computation - faster and uses less memory
- `.argmax(dim=1)` picks the class with the highest score for each image

> **Bug fixed:** The original looped as `for im, lab in test_loader` but then wrote
> `im, la = im.to(device), la.to(device)` - using the old `la` from training instead of `lab`.
> This meant evaluation was comparing predictions against the wrong labels.

In [ ]:
model.eval()  # Disable Dropout for evaluation
correct = 0
total   = 0

with torch.no_grad():
    for images, labels in test_loader:  # BUG FIX: original used 'lab' then accidentally used 'la'
        images, labels = images.to(device), labels.to(device)

        predicted = model(images).argmax(dim=1)  # Pick class with highest score
        correct += (predicted == labels).sum().item()
        total   += labels.size(0)

print(f'Test Accuracy: {100 * correct / total:.2f}%')
print(f'Correctly classified {correct} out of {total} images')